# Losses caused by the bench (position-aware)

Weeks a team lost, but would have won if it had started benched players who played on days when a
starting slot was empty, **counting only moves the lineup rules allow**.

For each team and day:
1. Take every player who played that day: starters and bench (IR excluded).
2. Find the best legal lineup using each player's eligible slots *on that day* (from ESPN's daily
   data) and the league's starting slots that season. Every starter who played stays in the lineup,
   though he may move to another slot he qualifies for. Bench players then fill as many open slots
   as possible, best first.
3. Bench players who make that lineup are "left on the bench". Their stats are added to the team's
   week, and the week is scored again against the opponent's actual totals.

Starters who played are never benched, so this counts empty slots, not starting the wrong player.
Best bench player = most fantasy points (points seasons) or most PTS+REB+AST+3PM+STL+BLK−TO
(category seasons). Works for points seasons (2020–21) and category seasons (2022–).

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from fbb.db import engine, read_query

SEASON = None    # e.g. 2027 for one season, None for every season with daily data

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", None)    # show full player lists
COUNT_STATS = ["PTS", "REB", "AST", "3PM", "STL", "BLK", "TO", "FGM", "FGA", "FTM", "FTA"]

## League settings: starting slots and categories each season

In [2]:
seasons = pd.read_sql("SELECT season, scoring_type FROM core.seasons", engine)
scoring_type = seasons.set_index("season").scoring_type

slot_counts = pd.read_sql("""
    SELECT l.season, s.key::int AS slot_id, s.value::int AS n
    FROM raw.espn_league l
    CROSS JOIN LATERAL jsonb_each_text(l.payload #> '{settings,rosterSettings,lineupSlotCounts}') s
    WHERE l.kind = 'league' AND s.key::int NOT IN (12, 13) AND s.value::int > 0   -- 12 = BE, 13 = IR
    ORDER BY 1, 2
""", engine)
slot_names = pd.read_sql("SELECT slot_id, slot_name FROM ref.lineup_slots", engine).set_index("slot_id").slot_name

# every starting slot each season, one entry per slot (e.g. UT twice)
lineup = {season: [sid for sid, n in zip(g.slot_id, g.n) for _ in range(n)]
          for season, g in slot_counts.groupby("season")}

cats = pd.read_sql("SELECT season, stat_name, lower_is_better FROM core.scoring_items", engine)
cats_by_season = {s: list(zip(g.stat_name, g.lower_is_better)) for s, g in cats.groupby("season")}

pd.DataFrame({"scoring": scoring_type,
              "starting slots": pd.Series({s: " ".join(slot_names[x] for x in v) for s, v in lineup.items()})})

,scoring,starting slots
2020,H2H_POINTS,PG SG SF PF C C G F UT UT UT
2021,H2H_POINTS,PG SG SF PF C C G F UT UT
2022,H2H_MOST_CATEGORIES,PG SG SF PF C UT UT
2023,H2H_MOST_CATEGORIES,PG SG SF PF C UT UT
2024,H2H_MOST_CATEGORIES,PG SG SF PF C UT UT
2025,H2H_MOST_CATEGORIES,PG SG SF PF C UT UT
2026,H2H_MOST_CATEGORIES,PG SG SF PF C UT UT
2027,H2H_MOST_CATEGORIES,PG SG SF PF C UT UT


## Players who played on days a team could have used them

Only team-days with at least one benched player who played. Eligible slots come from the raw daily
ESPN data, so they're the positions each player had *that day*.

In [3]:
season_filter = "" if SEASON is None else f"AND pd.season = {int(SEASON)}"
stat_cols = ",\n       ".join(f"coalesce((pd.stats->>'{s}')::numeric, 0) AS \"{s}\"" for s in COUNT_STATS)

cand = pd.read_sql(f"""
WITH elig AS (
    SELECT r.season, r.scoring_period,
           (m -> side ->> 'teamId')::int AS team_id,
           (e ->> 'playerId')::int       AS player_id,
           ARRAY(SELECT x::int FROM jsonb_array_elements_text(e #> '{{playerPoolEntry,player,eligibleSlots}}') x)
                                          AS eligible_slots
    FROM raw.espn_scoring_periods r
    CROSS JOIN LATERAL jsonb_array_elements(r.payload -> 'schedule') m
    CROSS JOIN LATERAL unnest(ARRAY['home', 'away']) side
    CROSS JOIN LATERAL jsonb_array_elements(m -> side #> '{{rosterForCurrentScoringPeriod,entries}}') e
    WHERE m ? side
),
pd AS (
    SELECT *, count(*) FILTER (WHERE lineup_slot = 'BE' AND played)
                  OVER (PARTITION BY season, scoring_period, team_id) AS bench_played
    FROM core.player_days
)
SELECT pd.season, pd.scoring_period, pd.matchup_id, pd.team_id, pd.player_id, pd.player_name,
       pd.lineup_slot, pd.is_starter, pd.played, pd.fantasy_points, el.eligible_slots,
       {stat_cols}
FROM pd
JOIN elig el USING (season, scoring_period, team_id, player_id)
WHERE (pd.played OR pd.is_starter) AND pd.bench_played > 0 AND pd.lineup_slot <> 'IR' {season_filter}
""", engine)

points_season = cand.season.map(scoring_type).eq("H2H_POINTS")
cand["value"] = np.where(points_season, cand.fantasy_points,
                         cand.PTS + cand.REB + cand.AST + cand["3PM"] + cand.STL + cand.BLK - cand.TO)
cand["eligible_slots"] = cand.eligible_slots.map(set)
print(f"{len(cand):,} player-days on {cand.groupby(['season', 'scoring_period', 'team_id']).ngroups:,} team-days")

53,246 player-days on 5,270 team-days


## Best legal lineup each day

A matching problem: players × starting slots, solved with `linear_sum_assignment`. Weights:
a starter who played is worth far more than anything else, so he always stays in; a bench player is
worth a fixed bonus plus his value, so more filled slots always beats one better player; a slot a
player isn't eligible for is forbidden. Extra "not in the lineup" columns let any player sit.

In [4]:
STARTER, BENCH, IDLE, FORBIDDEN = 1e6, 1e3, 1, -1e9

def bench_fill(day, slots):
    """Best legal lineup over `slots` for one team-day.
    Returns (bench players who make it, starters who didn't play and are pushed out of it,
    number of slots that were empty and get filled)."""
    n, k = len(day), len(slots)
    w = np.zeros((n, k + n))            # columns k.. are "not in the lineup" (weight 0)
    w[:, :k] = FORBIDDEN
    starter, played = day.is_starter.to_numpy(), day.played.to_numpy()
    for i, (elig, val) in enumerate(zip(day.eligible_slots, day.value)):
        weight = STARTER if starter[i] and played[i] else IDLE if starter[i] else BENCH + val
        for j, s in enumerate(slots):
            if s in elig:
                w[i, j] = weight
    rows, cols = linear_sum_assignment(w, maximize=True)
    in_lineup = {r for r, c in zip(rows, cols) if c < k and w[r, c] > FORBIDDEN}
    bench = [i for i in range(n) if not starter[i] and i in in_lineup]
    out = [i for i in range(n) if starter[i] and not played[i] and i not in in_lineup]
    out = out[:len(bench)]              # one pushed-out starter per bench player at most
    return day.iloc[bench], day.iloc[out], len(bench) - len(out)

added, replaced = [], []
for (season, period, team), day in cand.groupby(["season", "scoring_period", "team_id"]):
    slots = lineup[season]
    if (day.is_starter & day.played).sum() >= len(slots):   # every slot had a player who played
        continue
    bench, out, empty = bench_fill(day, slots)
    if bench.empty:
        continue
    added.append(bench)
    out = out[["season", "matchup_id", "team_id", "player_name"]]
    if empty:   # bench players who'd fill a slot nobody was in
        filler = pd.DataFrame([{"season": season, "matchup_id": day.matchup_id.iloc[0],
                                "team_id": team, "player_name": "(empty slot)"}] * empty)
        out = pd.concat([out, filler])
    replaced.append(out)
added = pd.concat(added, ignore_index=True)
replaced = pd.concat(replaced, ignore_index=True)
print(f"{len(added):,} bench games could legally have filled a slot "
      f"({(replaced.player_name == '(empty slot)').sum():,} of them a slot nobody was in)")

3,186 bench games could legally have filled a slot (328 of them a slot nobody was in)


## Score each week again with the bench players added

In [5]:
tm = pd.read_sql("""
    SELECT tm.season, tm.matchup_id, tm.matchup_period, tm.is_playoff, tm.team_id, tm.opponent_id,
           tm.points, tm.opponent_points, tm.result
    FROM core.team_matchups tm
    JOIN core.seasons s ON s.season = tm.season
    WHERE tm.result IS NOT NULL AND tm.opponent_id IS NOT NULL          -- finished, not a bye
      AND tm.season IN (SELECT DISTINCT season FROM core.player_days)  -- seasons with daily data
      -- 2020 weeks 20-23: NBA shut down for COVID, every score 0-0, ESPN still gave out wins
      AND NOT (s.scoring_type = 'H2H_POINTS' AND tm.points = 0 AND tm.opponent_points = 0)
""", engine)
if SEASON is not None:
    tm = tm[tm.season == SEASON]

totals = (pd.read_sql("SELECT season, matchup_id, team_id, stat_name, value FROM core.matchup_categories", engine)
            .pivot_table(index=["season", "matchup_id", "team_id"], columns="stat_name", values="value", aggfunc="max"))

key = ["season", "matchup_id", "team_id"]
add_wk = added.groupby(key)[COUNT_STATS + ["fantasy_points"]].sum()
add_wk["games_left_on_bench"] = added.groupby(key).size()
def name_counts(s):
    """'A (3), B, C (2)': each name with how many games, most first."""
    return ", ".join(f"{n} ({k})" if k > 1 else n for n, k in s.value_counts().items())

add_wk["players"] = added.groupby(key).player_name.agg(name_counts)
add_wk["started_instead"] = replaced.groupby(key).player_name.agg(name_counts)
NO_ADD = pd.Series(0.0, index=COUNT_STATS + ["fantasy_points"])

def cat_value(t, cat):
    if cat == "FG%":
        return t["FGM"] / t["FGA"] if t["FGA"] else np.nan
    if cat == "FT%":
        return t["FTM"] / t["FTA"] if t["FTA"] else np.nan
    return t[cat]

def score(w, add):
    """W/L/T for team-matchup `w` with `add` (bench stats) added to the team's side."""
    if scoring_type[w.season] == "H2H_POINTS":
        mine, theirs = w.points + add["fantasy_points"], w.opponent_points
        return "W" if mine > theirs else "L" if mine < theirs else "T"
    me  = totals.loc[(w.season, w.matchup_id, w.team_id)].fillna(0)
    opp = totals.loc[(w.season, w.matchup_id, w.opponent_id)].fillna(0)
    me  = me.add(add.reindex(me.index).fillna(0))
    wins = losses = 0
    for cat, low in cats_by_season[w.season]:
        a, b = cat_value(me, cat), cat_value(opp, cat)
        if pd.isna(a) or pd.isna(b) or a == b:
            continue
        if (a > b) != low: wins += 1
        else: losses += 1
    return "W" if wins > losses else "L" if wins < losses else "T"

rows = []
for w in tm.itertuples():
    k = (w.season, w.matchup_id, w.team_id)
    add = add_wk.loc[k] if k in add_wk.index else None
    rows.append({
        "check": score(w, NO_ADD),                                   # should equal the real result
        "with_bench": score(w, NO_ADD if add is None else add[NO_ADD.index]),
        "games_left_on_bench": 0 if add is None else int(add.games_left_on_bench),
        "players": "" if add is None else add.players,
        "started_instead": "" if add is None else add.started_instead,
    })
weeks = pd.concat([tm.reset_index(drop=True), pd.DataFrame(rows)], axis=1)

# Check: with nothing added, the scoring must reproduce every real result. Weeks it can't
# (e.g. 2022 week 22, where ESPN has no category totals) are left out, since they can't be re-scored.
mismatch = weeks[weeks.check != weeks.result]
weeks = weeks[weeks.check == weeks.result].copy()
print(f"{len(weeks):,} team-weeks re-scored; {len(mismatch)} left out because re-scoring can't reproduce ESPN's result")
mismatch[["season", "matchup_period", "team_id", "opponent_id", "result", "check"]]

1,660 team-weeks re-scored; 4 left out because re-scoring can't reproduce ESPN's result


,season,matchup_period,team_id,opponent_id,result,check
390,2022,22,12,9,L,T
392,2022,22,7,10,L,T
533,2022,22,9,12,W,T
535,2022,22,10,7,W,T


## Results by team (all seasons, most recent team name)

In [6]:
teams = pd.read_sql("""
    SELECT DISTINCT ON (t.team_id) t.team_id, t.team_name, m.display_name AS manager
    FROM core.teams t LEFT JOIN core.members m ON m.member_id = t.primary_owner_id
    ORDER BY t.team_id, t.season DESC
""", engine)

weeks["lost_to_bench"] = weeks.result.eq("L") & weeks.with_bench.eq("W")
weeks["lost_to_tie"]   = weeks.result.eq("L") & weeks.with_bench.eq("T")

summary = (weeks.groupby("team_id")
           .agg(seasons=("season", "nunique"),
                losses=("result", lambda r: (r == "L").sum()),
                losses_to_bench=("lost_to_bench", "sum"),
                losses_to_tie=("lost_to_tie", "sum"),
                games_left_on_bench=("games_left_on_bench", "sum"))
           .reset_index()
           .merge(teams, on="team_id")
           .sort_values(["losses_to_bench", "games_left_on_bench"], ascending=False))
summary[["team_name", "manager", "seasons", "losses", "losses_to_bench", "losses_to_tie", "games_left_on_bench"]]

,team_name,manager,seasons,losses,losses_to_bench,losses_to_tie,games_left_on_bench
11,Karaban Caravan,espn76571611,7,89,13,1,480
7,Screwston Sweats,jaredsux420,8,65,11,1,247
9,Team Youngbois😎,ChrisPaul462,7,71,10,1,294
8,Edgecombing on my Giannis,Will43329,7,69,10,1,223
4,Naz Reid,espn66314636,8,71,10,0,210
6,Jons of Soe Ingles,ESPNFAN3868298591,8,84,7,0,270
5,H-Tine Astromechs,Maverikrulz,8,63,7,3,258
3,Ligma Sugma,Jared Sweat,8,74,6,3,290
0,TWIN FADE AWAY,ESPNFAN72509299,8,53,6,0,250
2,Ant's Unwanted Children,Smbeast98,8,54,6,2,220


## Losses to the bench by season

In [7]:
(weeks.merge(teams, on="team_id")
      .pivot_table(index="team_name", columns="season", values="lost_to_bench", aggfunc="sum", margins=True, margins_name="total")
      .sort_values("total", ascending=False))

season,2020,2021,2022,2023,2024,2025,2026,2027,total
team_name,,,,,,,,,
total,12.0,11.0,20.0,20.0,8.0,11.0,10.0,0.0,92
Karaban Caravan,NaN,2.0,5.0,5.0,0.0,1.0,0.0,0.0,13
Screwston Sweats,4.0,0.0,1.0,0.0,1.0,2.0,3.0,0.0,11
Edgecombing on my Giannis,NaN,1.0,4.0,2.0,1.0,0.0,2.0,0.0,10
Team Youngbois😎,NaN,1.0,3.0,5.0,1.0,0.0,0.0,0.0,10
Naz Reid,3.0,1.0,3.0,1.0,0.0,1.0,1.0,0.0,10
H-Tine Astromechs,3.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,7
Jons of Soe Ingles,0.0,1.0,2.0,1.0,2.0,0.0,1.0,0.0,7
Ant's Unwanted Children,0.0,1.0,1.0,1.0,2.0,0.0,1.0,0.0,6


## Every week lost to the bench, and who was left there

In [8]:
(weeks[weeks.lost_to_bench]
      .merge(teams, on="team_id")
      .merge(teams[["team_id", "team_name"]].rename(columns={"team_id": "opponent_id", "team_name": "opponent"}), on="opponent_id")
      .sort_values(["season", "matchup_period"], ascending=False)
      .rename(columns={"players": "left_on_bench"})
      [["season", "matchup_period", "team_name", "opponent", "games_left_on_bench", "left_on_bench", "started_instead"]])

,season,matchup_period,team_name,opponent,games_left_on_bench,left_on_bench,started_instead
82,2026,20,Edgecombing on my Giannis,Ligma Sugma,15,"Cameron Johnson (3), Quentin Grimes (2), Kyle Filipowski (2), Nickeil Alexander-Walker (2), Naji Marshall (2), Rudy Gobert (2), Draymond Green, Giannis Antetokounmpo","Herbert Jones (3), Nikola Vucevic (3), Jrue Holiday (2), Luke Kornet (2), Jay Huff (2), Jerami Grant (2), Davion Mitchell"
85,2026,20,Screwston Sweats,Karaban Caravan,19,"Shai Gilgeous-Alexander (3), Tobias Harris (3), Jabari Smith Jr. (3), Jaime Jaquez Jr. (3), Bub Carrington (3), Mitchell Robinson (3), Zeke Nnaji","Jarrett Allen (7), Josh Hart (4), Jaylon Tyson (4), Derrick White (2), Jalen Brunson, Jock Landale"
86,2026,20,Jons of Soe Ingles,Seattle SuperSonics,17,"Moussa Diabate (3), Scoot Henderson (3), Stephon Castle (3), Nolan Traore (2), Dylan Harper (2), Trey Murphy III (2), P.J. Washington (2)","Anthony Black (7), Karl-Anthony Towns (3), Mikal Bridges (3), De'Aaron Fox (2), Desmond Bane, Austin Reaves"
83,2026,19,Edgecombing on my Giannis,Jons of Soe Ingles,3,"Jerami Grant, Giannis Antetokounmpo, Naji Marshall","VJ Edgecombe (2), Nikola Vucevic"
84,2026,19,Screwston Sweats,Seattle SuperSonics,19,"Jaime Jaquez Jr. (4), Zeke Nnaji (3), Tobias Harris (3), Bub Carrington (2), Andrew Wiggins (2), Shai Gilgeous-Alexander (2), Jabari Smith Jr. (2), Mitchell Robinson","Jarrett Allen (5), Jaylon Tyson (5), Bobby Portis (3), Derrick White (2), Jock Landale (2), Josh Hart, Jalen Brunson"
87,2026,19,H-Tine Astromechs,Naz Reid,1,Brook Lopez,Daniel Gafford
88,2026,17,Ant's Unwanted Children,Naz Reid,19,"Matas Buzelis (2), DeMar DeRozan (2), Nikola Jokic (2), Onyeka Okongwu (2), Bilal Coulibaly (2), Derik Queen (2), Moses Moody, Brandin Podziemski, Dyson Daniels, Kyle Kuzma, Anthony Edwards, Isaiah Collier, Devin Vassell","(empty slot) (18), Devin Booker"
89,2026,16,Screwston Sweats,H-Tine Astromechs,3,"Jock Landale (2), Tobias Harris","Bobby Portis, Jarrett Allen, Shai Gilgeous-Alexander"
90,2026,14,Naz Reid,Screwston Sweats,2,"Sam Hauser, Malik Monk","Santi Aldama, Cooper Flagg"
91,2026,4,Zebras,Edgecombing on my Giannis,8,"Herbert Jones, Deandre Ayton, Domantas Sabonis, Payton Pritchard, Coby White, Keyonte George, Deni Avdija, Brandin Podziemski","(empty slot) (6), Coby White, Brandon Ingram"


## How much positions matter

The SQL version (`sql/analysis/bench_losses.sql`) covers category seasons only, so this compares the
same seasons. Two differences:
- The SQL ignores positions, which makes its numbers higher.
- The SQL only fills slots whose starter didn't play. This notebook also fills slots that were left
  completely empty (no one in them), which can make its numbers higher.

In [9]:
cat_weeks = weeks[weeks.season.map(scoring_type).ne("H2H_POINTS")]
here = (cat_weeks.groupby("team_id")
        .agg(losses_to_bench=("lost_to_bench", "sum"), games_left_on_bench=("games_left_on_bench", "sum"))
        .reset_index().merge(teams, on="team_id"))
sql_version = read_query("bench_losses.sql", **({} if SEASON is None else {"season": SEASON}))
(here.merge(sql_version[["team_name", "losses_to_bench", "games_left_on_bench"]], on="team_name", suffixes=("", "_ignoring_positions"))
     [["team_name", "losses_to_bench", "losses_to_bench_ignoring_positions", "games_left_on_bench", "games_left_on_bench_ignoring_positions"]]
     .sort_values("losses_to_bench", ascending=False))

,team_name,losses_to_bench,losses_to_bench_ignoring_positions,games_left_on_bench,games_left_on_bench_ignoring_positions
11,Karaban Caravan,11,13,464,513.0
8,Edgecombing on my Giannis,9,9,176,221.0
9,Team Youngbois😎,9,9,276,321.0
7,Screwston Sweats,7,8,168,194.0
4,Naz Reid,6,7,155,173.0
6,Jons of Soe Ingles,6,9,203,270.0
0,TWIN FADE AWAY,6,7,202,172.0
2,Ant's Unwanted Children,5,5,165,138.0
5,H-Tine Astromechs,3,5,159,195.0
1,Zebras,3,7,125,174.0
